# 03 - Baselines y modelos

Se entrenan dos modelos y se comparan por **PR-AUC** en validación. El PR-AUC (área bajo la curva precision-recall) mide
qué tan bien el modelo **ordena** los casos poniendo los fraudes arriba, y es
robusto al desbalance. Es independiente del umbral, por lo que sirve para
comparar la calidad intrínseca de los modelos antes de calibrar y decidir el
umbral (eso se hace en el notebook 04).

In [1]:
import sys
sys.path.append("..")
%load_ext autoreload
%autoreload 2

from src.data import load_raw, clean
from src.features import temporal_split, build_preprocessor, NUM_FEATURES, CAT_LOW, CAT_HIGH
from src.model import build_logistic, build_xgb
from sklearn.metrics import average_precision_score

# Recargar, limpiar y partir
df = clean(load_raw("../data/dataset.csv"))
train, val, test = temporal_split(df, train_frac=0.7, val_frac=0.15)
RANDOM_STATE = 42

FEATURES = NUM_FEATURES + CAT_LOW + CAT_HIGH
X_train, y_train = train[FEATURES], train["fraude"]
X_val,   y_val   = val[FEATURES],   val["fraude"]

# Peso para compensar el desbalance (lo usan XGBoost y Optuna)
spw = (y_train == 0).sum() / (y_train == 1).sum()

## Regresión Logística

Modelo lineal, interpretable, que sirve de linea base. Su pipeline incluye:

- **`StandardScaler`**: la regresión logística es sensible a la escala de las variables,
  así que se estandarizan (media 0, desvío 1).
- **`class_weight="balanced"`**: maneja el desbalance dándole más peso a la
  clase minoritaria (fraude), inversamente a su frecuencia. Sin esto, el modelo
  tendería a ignorar los fraudes.
- **`max_iter=1000`**: asegura que el optimizador tenga iteraciones suficientes
  para converger.

In [2]:
# Entrenar el baseline
logistic = build_logistic()
logistic.fit(X_train, y_train)

# Primer chequeo con PR-AUC en validación
p_val = logistic.predict_proba(X_val)[:, 1]
print("PR-AUC (val):", round(average_precision_score(y_val, p_val), 4))

PR-AUC (val): 0.3138


## XGBoost

Captura no linealidades e interacciones que la regresión logística no puede. No
lleva escalado (los árboles son invariantes a la escala). Hiperparámetros:

- **`scale_pos_weight=18.3`**: maneja el desbalance (negativos/positivos ≈ 95/5).
  Es el equivalente al `class_weight` de la regresión logística.
- **`n_estimators=400`** y **`learning_rate=0.05`**: cuántos árboles y cuánto
  aporta cada uno. Van de la mano: learning rate bajo pide más árboles, y da un
  aprendizaje más gradual y robusto.
- **`max_depth=6`**: profundidad de cada árbol; controla la complejidad y el
  riesgo de sobreajuste.
- **`subsample=0.8`** y **`colsample_bytree=0.8`**: cada árbol usa el 80% de las
  filas y el 80% de las columnas. Ese azar reduce el sobreajuste.
- **`eval_metric="aucpr"`**: la métrica que XGBoost monitorea internamente
  (PR-AUC, coherente con la métrica de comparación).

In [3]:
from src.model import build_xgb

xgb = build_xgb(scale_pos_weight=spw)
xgb.fit(X_train, y_train)

p_val_xgb = xgb.predict_proba(X_val)[:, 1]
print("PR-AUC XGBoost (val):", round(average_precision_score(y_val, p_val_xgb), 4))

PR-AUC XGBoost (val): 0.4352


## XGBoost optimizado con Optuna

Se afinan los hiperparámetros de XGBoost con **Optuna**, que busca la combinación que maximiza el PR-AUC en
validación. Para acelerar, se preprocesa una sola vez (fit en train, transform
en val) y cada prueba solo reentrena el clasificador. La búsqueda se centra en
los hiperparámetros que más influyen: los que controlan el aprendizaje
(`n_estimators`, `learning_rate`), la complejidad de los árboles (`max_depth`,
`min_child_weight`, `gamma`) y la regularización estocástica (`subsample`,
`colsample_bytree`).

In [4]:
import optuna
from xgboost import XGBClassifier
from sklearn.metrics import average_precision_score

optuna.logging.set_verbosity(optuna.logging.WARNING)  # menos ruido en la salida

# Preprocesar una sola vez (fit en train) — correcto y mucho más rápido
pre = build_preprocessor()
X_train_p = pre.fit_transform(X_train, y_train)
X_val_p   = pre.transform(X_val)

def objetivo(trial):
    params = {
        "n_estimators":     trial.suggest_int("n_estimators", 200, 800),
        "learning_rate":    trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        "max_depth":        trial.suggest_int("max_depth", 3, 8),
        "subsample":        trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "gamma":            trial.suggest_float("gamma", 0.0, 5.0),
        "reg_lambda":       trial.suggest_float("reg_lambda", 0.1, 10.0, log=True),
    }
    clf = XGBClassifier(**params, scale_pos_weight=spw, eval_metric="aucpr",
                        random_state=RANDOM_STATE, n_jobs=-1)
    clf.fit(X_train_p, y_train)
    p = clf.predict_proba(X_val_p)[:, 1]
    return average_precision_score(y_val, p)

estudio = optuna.create_study(direction="maximize",
                              sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
estudio.optimize(objetivo, n_trials=30, show_progress_bar=True)

print("Mejor PR-AUC (val):", round(estudio.best_value, 4))
print("Mejores hiperparámetros:", estudio.best_params)

  0%|          | 0/30 [00:00<?, ?it/s]

Mejor PR-AUC (val): 0.4422
Mejores hiperparámetros: {'n_estimators': 620, 'learning_rate': 0.03013644571592144, 'max_depth': 4, 'subsample': 0.689589490361874, 'colsample_bytree': 0.9946485338702366, 'min_child_weight': 7, 'gamma': 0.9262986969204228, 'reg_lambda': 0.33851798850320963}


### Resultado del tuning

Optuna mejora el PR-AUC en validación de 0,427 (base) a **0,442** (~3,5%). La
mejora es modesta: el modelo base ya captura la mayor parte de la señal.

Los mejores hiperparámetros describen un modelo **más regularizado**: árboles
más superficiales (`max_depth=4` vs 6), más restrictivos (`min_child_weight=7`,
`gamma=0,93`) y más árboles con learning rate más bajo. Sugiere que el modelo
generaliza mejor con menos complejidad por árbol.

El PR-AUC en val es optimista (Optuna optimizó sobre val); la evaluación honesta
es en test (notebook 04), comparando la ganancia del modelo base vs. optimizado.

## Comparación

In [ ]:
from sklearn.metrics import average_precision_score

spw = (y_train == 0).sum() / (y_train == 1).sum()

modelos = {
    "Regresión logística": build_logistic(),
    "XGBoost": build_xgb(scale_pos_weight=spw),
}

resultados = {}
for nombre, modelo in modelos.items():
    modelo.fit(X_train, y_train)
    p = modelo.predict_proba(X_val)[:, 1]
    resultados[nombre] = average_precision_score(y_val, p)
    print(f"{nombre:22} PR-AUC (val): {resultados[nombre]:.4f}")